# u4lib — Python tutorial

Bindings for the header-only C++ **u4lib** — Wigner-Racah algebra for U(4)/SU(4),
the Wigner **supermultiplet** symmetry (spin-isospin SU(4) ⊃ SU(2)_S × SU(2)_T).
Names mirror the C++ API 1:1. Build first with `make python`; the bindings are
in [pyu4lib.cpp](pyu4lib.cpp).

| Python entry point | Purpose |
|---|---|
| `su4.irrep(P1,P2,P3)`, `irrep(P1,P2,P3)`, `irrep(n1,n2,n3,n4)` | SU(4)/U(4) irreps |
| `gelfand(...)` | Gelfand–Tsetlin state (10 labels) + generator MEs |
| `physicalvector(irrep, SS, TT)` | physical (S,T) state |
| `Eta(a, b, c)` | Coupling multiplicity of c in a×b |
| `decomposition(a, b)` | Product irreps with their Eta multiplicities |
| `Zeta(irrep, SS, TT)` | (S,T) branching multiplicity (SS=2S, TT=2T) |
| `physicalBasis`, `gelfandBasis` | basis state enumerators |
| `wigU4U3db().get(...)` | U(4) ⊃ U(3) coefficients (2-D numpy) |
| `wigU4SU2SU2db().get_U4SU2SU2(...)` | U(4) ⊃ SU(2)×SU(2), structured |
| `u4u6db()/u4z6db()/u4u9db().get_U4*(...)` | 6-/9-irrep recoupling |

> **Asserts are compiled in** (no `-DNDEBUG`): indexing an empty/forbidden
> coupling `abort()`s the kernel. Guard with `.empty()` or check `Eta`/`Zeta`.

## 1. Import + initialize
`initialize()` cascades u4 → u3 → qcx_utils (Wigner backend + all DBs).

In [ ]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import Markdown, display

candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base, base / "python", base / "u4lib" / "python"))
python_dir = next((p.resolve() for p in candidates
                   if (p / "pyu4lib.cpp").is_file()), None)
if python_dir is None:
    raise FileNotFoundError("Launch Jupyter from QCX, u4lib, or u4lib/python.")
sys.path.insert(0, str(python_dir))
import u4lib as u4
np.set_printoptions(precision=10, suppress=True)
print("Extension:", u4.__file__)

def table(headers, rows):
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, r)) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

u4.initialize()
print("u4lib session initialized.")

## 2. Irreps

- `su4.irrep(P1, P2, P3)` — an SU(4) irrep; `str()` → `(P1,P2,P3)`.
- `irrep(P1, P2, P3)` — a U(4) irrep from SU(4) labels (n44 = 0); `str()` → `[n14,n24,n34,n44]`.
- `irrep(n1, n2, n3, n4)` — a U(4) irrep from the four row labels.

Dimensions (spot-checks): fundamental `(1,0,0)` = **4**, `(2,0,0)` = **10**,
`(0,1,0)` = **6**, adjoint `(1,0,1)` = **15** (self-conjugate), `(0,0,0)` = **1**.

In [ ]:
labels = [(0,0,0), (1,0,0), (2,0,0), (0,1,0), (1,0,1)]
rows = []
for P in labels:
    ir = u4.irrep(*P)
    rows.append((f"({P[0]},{P[1]},{P[2]})", str(ir),
                 (ir.n14(), ir.n24(), ir.n34(), ir.n44()), ir.dim(), str(ir.conjugate()),
                 round(ir.C1(),6), round(ir.C2(),6), round(ir.C3(),6), round(ir.C4(),6)))
table(["SU(4) (P1,P2,P3)", "U(4) str", "(n14..n44)", "dim", "conjugate", "C1", "C2", "C3", "C4"], rows)

# Two ways to build the same U(4) irrep:
print("irrep(2,0,0)   ->", str(u4.irrep(2,0,0)))            # [2,0,0,0]
print("irrep(4,2,2,0) ->", str(u4.irrep(4,2,2,0)),
      " (P1,P2,P3) =", u4.irrep(4,2,2,0).P1(), u4.irrep(4,2,2,0).P2(), u4.irrep(4,2,2,0).P3())

adj = u4.su4.irrep(1,0,1)
print("adjoint (1,0,1): dim", adj.dim(), "| self-conjugate?", str(adj.conjugate()) == str(adj),
      "| C2", adj.C2(), "| C3", adj.C3(), "| C4", adj.C4())

## 3. Gelfand–Tsetlin states

A U(4) GT state with 10 labels `n14,n24,n34,n44; n13,n23,n33; n12,n22; n11`.
Exposes the betweenness `check()`, SU(2) labels (2Λ, 2M_Λ), and the U(4)
generator matrix elements (`e11`…`e44` and the multi-step raising/lowering ones).

In [ ]:
g = u4.gelfand(1,0,0,0, 1,0,0, 1,0, 1)   # a state of the fundamental (1,0,0)
print("state     :", str(g), "| repr", repr(g))
print("valid?    :", g.check())
print("P1,P2,P3  :", g.P1(), g.P2(), g.P3())
print("2*Lambda, 2*M_Lambda:", g.Lambdax2(), g.MLambdax2())
print("e11,e22,e33,e44:", g.e11(), g.e22(), g.e33(), g.e44())
print("conjugate :", str(g.conjugate()), "| phase", g.conjugate_phase())

## 4. Coupling multiplicity `Eta`

`Eta(a, b, c)` = multiplicity of `c` in `a × b` (u4's analog of u3's `Rho`).
Two overloads: SU(4) labels, or irrep objects.

Spot-check: `4 ⊗ 4 = 10 ⊕ 6` — i.e. `(1,0,0)×(1,0,0)` gives `(2,0,0)` and
`(0,1,0)`, each once, and the dimensions sum to 16.

In [ ]:
fund = u4.irrep(1,0,0)
outs = [(2,0,0), (0,1,0), (0,0,0)]
rows = []
for P in outs:
    c = u4.irrep(*P)
    e = u4.Eta(fund, fund, c)
    rows.append((f"({P[0]},{P[1]},{P[2]})", c.dim(), e, c.dim()*e))
table(["output", "dim", "Eta", "dim x Eta"], rows)
tot = sum(r[-1] for r in rows if r[0] != "(0,0,0)")
print("4 x 4: (2,0,0) + (0,1,0) dims =", tot, "= 16 = 4*4:", tot == 16)
print("Eta by labels == by objects:",
      u4.Eta(1,0,0, 1,0,0, 2,0,0), u4.Eta(fund, fund, u4.irrep(2,0,0)))

## 5. Tensor-product decomposition

`decomposition(a, b)` returns `(irreps, etamaxs)`: each normalized U(4)
output irrep paired with its outer multiplicity. The adjoint product
`15 × 15` contains the adjoint twice and has total dimension 225.


In [ ]:
adjoint = u4.irrep(1, 0, 1)
outputs, etamaxs = u4.decomposition(adjoint, adjoint)
channels = [(ir.P1(), ir.P2(), ir.P3(), eta)
            for ir, eta in zip(outputs, etamaxs)]
assert len(channels) == len(set((p1, p2, p3) for p1, p2, p3, _ in channels))
assert (1, 0, 1, 2) in channels
assert sum(ir.dim() * eta for ir, eta in zip(outputs, etamaxs)) == adjoint.dim() ** 2
print("15 x 15 output channels (P1, P2, P3, Eta):", channels)

# Verify every reported outer multiplicity against Eta(a,b,c).
assert all(u4.Eta(adjoint, adjoint, ir) == eta
           for ir, eta in zip(outputs, etamaxs))

## 6. Spin-isospin content: `Zeta` and the physical basis

`Zeta(irrep, SS, TT)` = multiplicity of the spin-isospin channel `(S,T)` in the
U(4) irrep, with **SS = 2S** and **TT = 2T** (doubled).

Spot-check: the fundamental `(1,0,0)` is the **nucleon** supermultiplet
`(S,T) = (½,½)` → `SS=TT=1`, `Zeta = 1`; the symmetric `(2,0,0)` ("10") contains
`(S,T) = (1,1)` and `(0,0)` → `(SS,TT) = (2,2)` and `(0,0)` (that is 9 + 1 = 10).

In [ ]:
print("Zeta((1,0,0), SS=1, TT=1) [nucleon (1/2,1/2)]:", u4.Zeta(1,0,0, 1,1))
print("physicalBasis((1,0,0)) (SS,TT,zeta):",
      [(v.SS(), v.TT(), v.zeta()) for v in u4.physicalBasis(u4.irrep(1,0,0))])

# physicalBasis as a table -- the (SS,TT,zeta) content of an irrep:
pb = u4.physicalBasis(u4.irrep(2,0,0))
table(["state", "SS (=2S)", "TT (=2T)", "zeta"],
      [(repr(v), v.SS(), v.TT(), v.zeta()) for v in pb])

# gelfandBasis size == dim of the irrep
gb = u4.gelfandBasis(u4.irrep(0,1,0))
print("gelfandBasis((3,1,0)):", len(gb), "states (= dim", u4.irrep(0,1,0).dim(), ")")
table(["state"], [(repr(g),) for g in gb])

## 7. U(4) ⊃ U(3) coefficients — `wigU4U3db`

`get(...)` takes, per irrep a/b/c, the seven U(4)⊃U(3) labels
`(n14, n24, n34, n44, n13, n23, n33)`, and returns a **2-D** numpy array indexed
`[rho, eta]` (rho = SU(3) coupling multiplicity, eta = U(4) coupling multiplicity).

In [ ]:
m = np.asarray(u4.wigU4U3db().get_U4U3(2,1,1,0,2,1,0 , 3,2,1,0,3,2,0 , 3,2,1,0,3,2,0))
print(m)

## 8. U(4) ⊃ SU(2)×SU(2) coefficients — `wigU4SU2SU2db`

Per irrep: `(P1, P2, P3, SS, TT)`. `get(...)` returns a flat numpy array;
`get_U4SU2SU2(...)` returns a structured **U4SU2SU2** object indexed
`(zetaA, zetaB, zetaC, eta)`.

In [ ]:
db = u4.wigU4SU2SU2db()
# scalar check:
sc = db.get_U4SU2SU2(0,0,0,0,0, 0,0,0,0,0, 0,0,0,0,0)
print("scalar (0,0,0): size", sc.size(), "data", sc.data())

obj = db.get_U4SU2SU2(1,1,1,2,2,  1,1,1,4,2,  1,2,1,4,2)
print("empty?", obj.empty(), "| shape (zA,zB,zC,eta) =",
      obj.zetaAmax(), obj.zetaBmax(), obj.zetaCmax(), obj.etamax())
print(".data() =", obj.data())
print(obj)

## 9. Recoupling — `u4u6db` / `u4z6db` / `u4u9db`

The 6-irrep U- and Z-coefficients and the 9-irrep coefficients. Each `get_U4*`
takes the relevant coupling **multiplicities** (from `Eta`) and returns a
structured object. A scalar leg keeps every multiplicity 1.

In [ ]:
A=u4.irrep(2,1,1,0); B=u4.irrep(2,1,1,0); C=u4.irrep(2,1,1,0)
AB=u4.irrep(3,1,0,0); BC=u4.irrep(2,1,1,0); AC=u4.irrep(2,2,0,0)
ABC=u4.irrep(3,1,0,0)

eAB,eAB_C = u4.Eta(A,B,AB), u4.Eta(AB,C,ABC)
eBC,eA_BC = u4.Eta(B,C,BC), u4.Eta(A,BC,ABC)
eAC,eAC_B = u4.Eta(A,C,AC), u4.Eta(AC,B,ABC)

u6 = u4.u4u6db().get_U4U6(A,B,ABC,C, AB,eAB,eAB_C, BC,eBC,eA_BC)
print("U4U6 size", u6.size())
print(u6)

z6 = u4.u4z6db().get_U4Z6(B,A,ABC,C, AB,eAB,eAB_C, AC,eAC,eAC_B)
print("U4Z6 size", z6.size())
print(z6)

In [ ]:
A=u4.irrep(2,1,1,0); B=u4.irrep(2,1,1,0); C=u4.irrep(2,1,1,0); D=u4.irrep(2,1,1,0)
AB=u4.irrep(2,1,1,0); CD=u4.irrep(2,1,1,0); ACi=u4.irrep(2,1,1,0); BD=u4.irrep(2,1,1,0)
ABCD=u4.irrep(0,0,0,0)
eAB,eCD = u4.Eta(A,B,AB), u4.Eta(C,D,CD)
eAC,eBD = u4.Eta(A,C,ACi), u4.Eta(B,D,BD)
eABCD,eACBD = u4.Eta(AB,CD,ABCD), u4.Eta(ACi,BD,ABCD)
u9 = u4.u4u9db().get_U4U9(A,B,C,D, AB,CD, ACi,BD, ABCD, eAC,eBD,eABCD, eAB,eCD,eACBD)
print("U4U9 size", u9.size())
print(u9)

## 10. Physical generators and the physical→canonical transform

`RME_S`/`RME_T`/`RME_U` return `(matrix, bra_zeta, ket_zeta)`; the
physical→canonical expansion `phys_to_cano(p, MMS, MMT, gArr)` returns
`(canoBasis, T)`. All spin/isospin projections are doubled (`MMS=2MS`,
`MMT=2MT`).

In [ ]:
import math

basis = u4.physicalBasis(u4.irrep(2, 0, 0))   # (S,T)=(1,1) [0], (0,0) [1]
st11 = basis[0]

# S and T on (S=1,T=1): sqrt(S(S+1)(2S+1)(2T+1)) = sqrt(18)
print("<(1,1)||S||(1,1)> =", u4.RME_S(st11, st11)[0][0, 0], "  sqrt(18) =", math.sqrt(18))
print("<(1,1)||T||(1,1)> =", u4.RME_T(st11, st11)[0][0, 0])
print("<(1,1)||U||(1,1)> =", u4.RME_U(st11, st11)[0][0, 0])

# physical -> canonical expansion at the highest weight (MMS=SS, MMT=TT)
ir = u4.irrep(2, 0, 0)
gArr = u4.gelfandBasis(ir)
p = u4.physicalvector(ir, 2, 2)
canoBasis, T = u4.phys_to_cano(p, 2, 2, gArr)
print("HW amplitude:", T.ravel(), " zeta columns orthonormal:", float((T.T @ T)[0, 0]))

## 11. Parallelism

`u4lib` is **not** parallelizable from Python threads. The extension holds the
GIL (so threads give no speed-up), and the C++ coefficient databases keep
per-thread state as OpenMP `threadprivate` storage keyed on
`omp_get_thread_num()`. A `threading` / `ThreadPoolExecutor` worker is a plain OS
thread, **not** an OpenMP thread, so calling `initialize_thread()` there aborts
the interpreter. Do not use Python threads with `u4lib`.

**Native parallelism is C++ / OpenMP.** Inside an OpenMP parallel region each
thread sets up and tears down its own database state; the `src/` example drivers
show the pattern:

```cpp
u4::initialize(&argc, &argv);            // once, on the main thread
#pragma omp parallel num_threads(4)
{
    u4::initialize_thread();             // per OpenMP thread
#pragma omp for
    for (int t = 0; t < work.size(); ++t) {
        auto blk = u4::wigU4SU2SU2db().get_U4SU2SU2(/* labels */);
        // use blk; guard shared I/O with #pragma omp critical
    }
    u4::finalize_thread();
}
u4::finalize();
```

**From Python, parallelize across _processes_, not threads** — each process has
its own interpreter, GIL and library state. Run it as a standalone script (in a
notebook, `spawn` cannot pickle a cell-defined worker, so this does not work in a
cell):

```python
# save as run_parallel.py, then: python run_parallel.py
import concurrent.futures as cf, multiprocessing as mp

def work(idx):
    import u4lib as u4
    u4.initialize()
    try:
        return idx, u4.wigU4SU2SU2db().get_U4SU2SU2(1, 0, 0, 1, 1,
                                                    1, 0, 0, 1, 1,
                                                    2, 0, 0, 2, 2)(0, 0, 0, 0)
    finally:
        u4.finalize()

if __name__ == "__main__":
    ctx = mp.get_context("spawn")
    with cf.ProcessPoolExecutor(max_workers=4, mp_context=ctx) as ex:
        print([f.result() for f in [ex.submit(work, i) for i in range(4)]])
```

## 12. Finish

In [ ]:
u4.finalize()
print("Session finalized. All tutorial checks passed.")